# Business Context

* A multi-specialty private hospital chain with three facilities is experiencing a widening gap between
appointment demand and capacity. 

* Patients are reporting longer wait times. Some departments have backlogs of weeks while others are running below 60% occupancy. 

* The Clinical Operations Director needs an analysis of where bottlenecks are occurring and which departments have capacity to absorb additional volume .

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"..\datasets\uk_private_clinic_no_show_2026.csv")
df.head(5)

,appointment_id,patient_id,age,gender,postcode_area,distance_to_clinic_km,clinic_id,clinic_type,appointment_type,consultation_fee_gbp,...,days_since_last_visit,has_chronic_condition_flag,sms_reminder_sent,reminder_hours_before,reminder_response,number_of_reminders_sent,estimated_revenue_loss_gbp,staff_idle_cost_gbp,slot_rebooked_flag,no_show
0,1,861,53,Male,B15,6.2,C1,Physiotherapy,New Patient,93,...,204,0,1,48,Confirmed,1,0,0.00,0,0
1,2,1295,32,Male,CF10,5.9,C3,GP,Follow-up,68,...,123,1,1,24,Confirmed,1,68,23.80,1,1
2,3,1131,18,Male,BS1,3.6,C2,Physiotherapy,Follow-up,104,...,9,0,1,24,Confirmed,1,0,0.00,0,0
3,4,1096,43,Female,M1,7.5,C4,GP,Follow-up,93,...,55,0,1,48,Confirmed,1,93,32.55,0,1
4,5,1639,21,Male,G1,5.5,C5,Dermatology,New Patient,182,...,105,0,0,24,No Reminder,1,182,63.70,1,1


In [3]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ' , '_')
df.columns

Index(['appointment_id', 'patient_id', 'age', 'gender', 'postcode_area',
       'distance_to_clinic_km', 'clinic_id', 'clinic_type', 'appointment_type',
       'consultation_fee_gbp', 'scheduled_date', 'appointment_date',
       'appointment_day_of_week', 'appointment_hour', 'days_in_advance',
       'is_weekend', 'is_peak_hour', 'patient_total_appointments',
       'patient_previous_noshow_count', 'patient_noshow_rate',
       'days_since_last_visit', 'has_chronic_condition_flag',
       'sms_reminder_sent', 'reminder_hours_before', 'reminder_response',
       'number_of_reminders_sent', 'estimated_revenue_loss_gbp',
       'staff_idle_cost_gbp', 'slot_rebooked_flag', 'no_show'],
      dtype='object')

In [4]:
df = df.rename(columns={
    'consultation_fee_gbp' : 'consultation_fee' , 
    'scheduled_date' : 'appo_book_date',
    'days_in_advance' : 'appo_wait_days',
    'patient_previous_noshow_count' : 'missed_appointment'
})

In [5]:
df.info()
print('=='*20)
print('null check')
print('=='*20)
print(df.isnull().sum())
print('=='*20)
print('duplicate check')
print('=='*20)
print(df.duplicated().sum())
print('=='*20)
print('data shape before cleaning')
print('=='*20)
print(df.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   appointment_id              10000 non-null  int64  
 1   patient_id                  10000 non-null  int64  
 2   age                         10000 non-null  int64  
 3   gender                      10000 non-null  object 
 4   postcode_area               10000 non-null  object 
 5   distance_to_clinic_km       10000 non-null  float64
 6   clinic_id                   10000 non-null  object 
 7   clinic_type                 10000 non-null  object 
 8   appointment_type            10000 non-null  object 
 9   consultation_fee            10000 non-null  int64  
 10  appo_book_date              10000 non-null  object 
 11  appointment_date            10000 non-null  object 
 12  appointment_day_of_week     10000 non-null  object 
 13  appointment_hour            1000

* to check appo_book_date and appointment_date quality

In [6]:
df['appo_book_date'] = pd.to_datetime(df['appo_book_date'])
df['appointment_date'] = pd.to_datetime(df['appointment_date'])

# Find records where scheduled_date is NOT before appointment_date (same date is ok)
invalid_dates = df[df['appo_book_date'] > df['appointment_date']]

# Summary report
print("=" * 60)
print("DATE QUALITY CHECK: scheduled_date < appointment_date")
print("=" * 60)
print(f"Total appointments      : {len(df):,}")
print(f"Invalid date records   : {len(invalid_dates):,}")
print(f"Percentage invalid     : {len(invalid_dates)/len(df)*100:.2f}%")
print("=" * 60)

# Show the invalid rows (if any)
if len(invalid_dates) > 0:
    display("\nSample of invalid records:")
    display(invalid_dates[['appointment_id', 'patient_id', 'appo_book_date', 
                         'appointment_date', 'day_gap']].head(10))
else:
    print("\nAll records satisfy the condition: scheduled_date < appointment_date")

DATE QUALITY CHECK: scheduled_date < appointment_date
Total appointments      : 10,000
Invalid date records   : 0
Percentage invalid     : 0.00%

All records satisfy the condition: scheduled_date < appointment_date


* to check consultation fee not to become <=0

In [7]:
# Find invalid consultation fees (zero or negative)
invalid_fees = df[df['consultation_fee'] <= 0]

# Summary report
print("=" * 60)
print("CONSULTATION FEE QUALITY CHECK")
print("=" * 60)
print(f"Total appointments          : {len(df):,}")
print(f"Invalid fees (≤ 0)          : {len(invalid_fees):,}")
print(f"Percentage invalid          : {len(invalid_fees)/len(df)*100:.2f}%")
print("=" * 60)


# Show invalid records (if any)
if len(invalid_fees) > 0:
    print("\nSample of invalid fee records:")
    print(invalid_fees[['appointment_id', 'patient_id', 'clinic_type', 
                        'appointment_type', 'consultation_fee']].head(10))
else:
    print("\nAll consultation fees are valid (greater than 0).")

CONSULTATION FEE QUALITY CHECK
Total appointments          : 10,000
Invalid fees (≤ 0)          : 0
Percentage invalid          : 0.00%

All consultation fees are valid (greater than 0).


* to check appo_wait_days = appo_book_date - appointment_date 

In [13]:
# Convert to datetime
df['appo_book_date'] = pd.to_datetime(df['appo_book_date'])
df['appointment_date'] = pd.to_datetime(df['appointment_date'])

# Calculate actual days difference
df['calculated_days_in_advance'] = (df['appointment_date'] - df['appo_book_date']).dt.days

# Find mismatches
mismatch = df[df['appo_wait_days'] != df['calculated_days_in_advance']]

# Report
print("=" * 65)
print("DATA QUALITY CHECK: days_in_advance calculation")
print("=" * 65)
print(f"Total records              : {len(df):,}")
print(f"Mismatch records           : {len(mismatch):,}")
print(f"Percentage mismatch        : {len(mismatch)/len(df)*100:.2f}%")
print("=" * 65)

if len(mismatch) > 0:
    print("\nSample of mismatched records:")
    print(mismatch[['appointment_id', 'appo_book_date', 'appointment_date', 
                    'appo_wait_days', 'calculated_days_in_advance']].head(10))
else:
    print("\nAll records are correct. appo_wait_days = appointment_date - appo_book_date")

DATA QUALITY CHECK: days_in_advance calculation
Total records              : 10,000
Mismatch records           : 0
Percentage mismatch        : 0.00%

All records are correct. appo_wait_days = appointment_date - appo_book_date


In [8]:
# remove white space from object 
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

In [9]:
# check unique values
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    vals = df[col].unique()
    print(f"\n{col} ({len(vals)} unique): {sorted(vals)}")


gender (2 unique): ['Female', 'Male']

postcode_area (10 unique): ['B15', 'BS1', 'CF10', 'E1', 'G1', 'L1', 'LS1', 'M1', 'N1', 'SW1A']

clinic_id (5 unique): ['C1', 'C2', 'C3', 'C4', 'C5']

clinic_type (4 unique): ['Dental', 'Dermatology', 'GP', 'Physiotherapy']

appointment_type (2 unique): ['Follow-up', 'New Patient']

appointment_day_of_week (7 unique): ['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']

reminder_response (4 unique): ['Cancelled', 'Confirmed', 'Ignored', 'No Reminder']


In [11]:
df.to_csv(r"..\datasets\clean_dataset.csv", index=False)